# Lesson 18 Lab — Serving INT4 with vLLM

**Puzzle:** If a checkpoint says AWQ or GPTQ, will vLLM necessarily run it efficiently on the current GPU?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

A vLLM service couples checkpoint format, quantization backend, model runner, scheduler, paged KV cache, CUDA graphs, request batching, and sampling. Linear-kernel latency is only one component.

### Core mechanism

Prefill cost grows with prompt work while decode repeatedly processes small token steps and reads KV cache. Continuous batching improves utilization by combining requests, but queueing changes time-to-first-token and tail latency.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "18-vllm-int4-serving"
device = require_cuda()
torch.manual_seed(2026 + 18)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

An INT4 backend can save weight memory and allow more concurrency yet be slower for batch-one shapes. Compatibility tables change with GPU generation and release.

### What this code tests

The lab records vLLM availability and uses PyTorch batch-shape timings only as a warning; it labels vLLM service throughput `not_measured`.

**Experiment:** Probe vLLM availability and benchmark a small PyTorch W4-dequantized matmul across batch sizes as a backend-independent shape warning.

**Declared evidence label:** `compatibility-probe`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
import importlib.util
w=torch.randn(2048,2048,device=device,dtype=torch.bfloat16); _,_,dq=symmetric_quantize(w,bits=4,group_size=128); dq=dq.bfloat16()
rows=[]
for batch in (1,8,32):
    x=torch.randn(batch,2048,device=device,dtype=torch.bfloat16)
    rows.append({"batch":batch,"bf16":cuda_benchmark(lambda:x@w.t(),warmup=4,repeats=15),
                 "reference_w4_dequant":cuda_benchmark(lambda:x@dq.t(),warmup=4,repeats=15)})
installed=importlib.util.find_spec("vllm") is not None
result=base_result(18,"compatibility-probe"); result.update({"vllm_installed":installed,"pytorch_shape_warning":rows,
    "vllm_service_benchmark":"not_measured","conclusion":"PyTorch shape timing was measured separately; vLLM service performance requires an installed server and load test."})


## 3. Inspect the evidence

The timing is labeled PyTorch GPU evidence. vLLM throughput remains `not_measured` when the package/server is absent.

### Acceptance and rollback gate

Pass format/hardware load, operator, quality, TTFT, TPOT/inter-token latency, throughput, p90/p99, peak memory, and sustained-concurrency gates with a frozen request distribution.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "PyTorch shape timing was measured separately; vLLM service performance requires an installed server and load test.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "compatibility-probe",
  "executed_at_utc": "2026-08-07T14:45:55+00:00",
  "lesson": 18,
  "pytorch_shape_warning": [
    {
      "batch": 1,
      "bf16": {
        "median_ms": 0.01952,
        "p90_ms": 0.02032,
        "repeats": 15,
        "samples_ms": [
          0.030528,
          0.028448,
          0.02032,
          0.019488,
          0.01984,
          0.01968,
          0.019488,
          0.019616,
          0.019456,
          0.01952,
          0.019872,
          0.0192,
          0.01904,
          0.019168,
          0.019488
        ],
        "warmup": 4
      },
      "reference_w4_dequant": {
        "median_m

## 4. Explain the result

Pass checkpoint-format, hardware, load, operator, quality, and service-load gates before adopting a vLLM INT4 path.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).